<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/units/en/unit0/w10-mcp-resources-prompts-llms/notebook.ipynb)


In [45]:
# Preflight: environment checks with a fix for anything missing. It never raises.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

try:
    from bootcamp_agent.preflight import preflight
except ImportError:
    if "google.colab" in sys.modules:
        # Colab starts in /content with no course in it, so fetch one. A shallow
        # clone of the COHORT repository, which is the public one; the source
        # repository is private and would ask this learner for credentials.
        import subprocess

        target = Path("/content/dev3pack")
        if not (target / "pyproject.toml").exists():
            print("Colab detected — fetching the course (about 20 seconds)…")
            subprocess.run(
                ["git", "clone", "-q", "--depth", "1",
                 "https://github.com/Gecko-Academy/dev3pack-cohort-2026-09.git", str(target)],
                check=True,
            )
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-e", str(target)], check=True
        )
        REPO_ROOT = target
        sys.path.insert(0, str(REPO_ROOT / "src"))
        import os

        os.chdir(REPO_ROOT)
        from bootcamp_agent.preflight import preflight

        print(f"ready — the course is at {REPO_ROOT}")
    else:
        print("❌ bootcamp_agent not importable -> in the repo root run: uv sync --group dev")
        print("   then pick the .venv kernel (or start Jupyter with: uv run jupyter lab)")
else:
    preflight(REPO_ROOT)

✅ Python 3.11 (need >= 3.11)
✅ kernel is the repo .venv
✅ corpus loads (6 documents)
✅ lane = fake (deterministic, offline)
ready. LIVE is the fake lane.


In [46]:
from bootcamp_agent.checks import check, review
from bootcamp_agent.hints import hint  # noqa: F401 - hint("w10-e1") when you want a nudge
import bootcamp_agent.week0_checks  # noqa: F401 — importing is what registers them

# Unit 10: Resources, prompts, and the LLM

**Week 0 · Course B, chapter 2 of 3 · about 60 minutes**

**Goal:** Add the other two MCP primitives to the server you wrote in unit 9, then put a model in the loop. Leave knowing what a resource is for, why a prompt's name is not its title, and what the five steps of a tool call are.

**Why it matters:** Session 5 builds the agent loop, and the loop in this notebook is its smallest honest version. Session 10 packages a skill, which is a prompt template plus a when-to-use, served the way this unit serves one. Session 13 points a client at a hosted surface and asks it to plan a real transaction, so refusal and clarification have to be answers, not failures.

Some cells below ship **broken on purpose**, marked `<------ EDIT THIS LINE`. Run them first and read what happens. The last one answers a question nobody could answer, which is the failure this unit exists to make visible.

**Offline, and honest about it.** The deck calls Claude through the Anthropic Messages API. This notebook calls `FakeLLM` from `bootcamp_agent.llm` through the same `complete(system, user)` seam, and the tool-call decision is made by a small router you can read. A real model is one env var away (`BOOTCAMP_PROVIDER`, see `SETUP.md`); the loop does not change when you swap it in. The server, the client, and the protocol are real.

## 1. A resource: read-only context, one thing per line

**Context.** A tool is something the model *does*. A resource is something that *is*: read-only
data, addressed by a URI, fetched by the application rather than chosen by the model. The
server's list of supported zones is the example, and it is a resource rather than a tool because
nothing about reading it is a decision.

The cell below writes a server with unit 9's tool and a new resource. The resource works. It
also hands back the twelve zones as one comma-joined line, which every reader then has to take
apart before it can use it.

**Instructions.**

1. Run both cells. The resource is listed and read, and you get one long line.
2. Change the join so the resource answers one zone per line.
3. Read the URI the client printed. In version 1 of the SDK it came back as
   `file://locations.txt/`, with a trailing slash. Version 2 does not add one.

**Expected output**

```
resources: ['file://locations.txt']
Africa/Abidjan
America/Halifax
...
Europe/Paris
✅ w10-e1 passed
```

In [47]:
import tempfile

WORKDIR = Path(tempfile.mkdtemp(prefix="mcp-unit10-"))
SERVER_PATH = WORKDIR / "timezone_server.py"
FIXTURE = (
    REPO_ROOT / "units" / "en" / "unit0" / "w09-mcp-first-server" / "fixtures" / "timezones.json"
)

# The server is assembled from pieces, the way the deck adds one primitive per
# lesson: "tools and resources from before...". Each piece is a raw string,
# because a source file that contains "\n" must keep the two characters.
SERVER_HEAD = r'''
"""A timezone converter, served over MCP: a tool, a resource and a prompt."""

import json
from datetime import datetime, timedelta, timezone
from pathlib import Path

from mcp.server.mcpserver import MCPServer

ZONES = json.loads(Path(r"__FIXTURE__").read_text(encoding="utf-8"))["zones"]

mcp = MCPServer("Timezone Converter")


def _zone(name: str):
    """The zone from the system database, or January's offset from the fixture."""
    try:
        from zoneinfo import ZoneInfo

        return ZoneInfo(name)
    except Exception:
        return timezone(timedelta(hours=ZONES[name]))
'''

TOOL = r'''

@mcp.tool()
def convert_timezone(date_time: str, from_timezone: str, to_timezone: str) -> str:
    """Convert a datetime from one timezone to another.

    Args:
        date_time: the datetime in ISO format, for example 2025-01-20T09:50:00
        from_timezone: the source zone, for example Europe/London
        to_timezone: the target zone, for example Europe/Lisbon

    Returns:
        A line naming the target zone and the converted datetime.
    """
    moment = datetime.fromisoformat(date_time).replace(tzinfo=_zone(from_timezone))
    converted = moment.astimezone(_zone(to_timezone)).isoformat()
    return f"Time in {to_timezone}: {converted}"
'''

RESOURCE = r'''

@mcp.resource("file://locations.txt")
def get_locations() -> str:
    """Every timezone this server can convert, one per line."""
    return "\n".join(sorted(ZONES))  # <------ EDIT THIS LINE
'''

MAIN = r'''

if __name__ == "__main__":
    mcp.run(transport="stdio")
'''


def write_server(*pieces):
    """Assemble the server file from the pieces given, and fill in the fixture path."""
    source = "".join(pieces).replace("__FIXTURE__", str(FIXTURE))
    SERVER_PATH.write_text(source, encoding="utf-8")


write_server(SERVER_HEAD, TOOL, RESOURCE, MAIN)
print(f"wrote {SERVER_PATH}")
print("one tool, one resource")

wrote /tmp/mcp-unit10-82a4so6p/timezone_server.py
one tool, one resource


In [48]:
import asyncio

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

LOCATIONS_URI = "file://locations.txt"


async def read_resource(uri):
    """List the resources, then read one by its URI."""
    params = StdioServerParameters(command=sys.executable, args=[str(SERVER_PATH)])
    async with stdio_client(params) as (reader, writer):
        async with ClientSession(reader, writer) as session:
            await session.initialize()
            listed = await session.list_resources()
            print(f"resources: {[str(item.uri) for item in listed.resources]}")
            read = await session.read_resource(uri)
            return read.contents[0].text


async def read_prompt(name, request):
    """List the prompts, then render one with the person's request."""
    params = StdioServerParameters(command=sys.executable, args=[str(SERVER_PATH)])
    async with stdio_client(params) as (reader, writer):
        async with ClientSession(reader, writer) as session:
            await session.initialize()
            listed = await session.list_prompts()
            print(f"the server lists: {[item.name for item in listed.prompts]}")
            print(f"you are asking for: {name!r}")
            found = next((item for item in listed.prompts if item.name == name), None)
            if found is None:
                # Nothing on the server answers to that name, so there is nothing
                # to render. Report what was asked for and let the check say why.
                return (name, name, "")
            rendered = await session.get_prompt(name, arguments={"timezone_request": request})
            return (found.name, found.title, rendered.messages[0].content.text)


async def call_tool(name, arguments):
    """Call one tool and return the text it answered with."""
    params = StdioServerParameters(command=sys.executable, args=[str(SERVER_PATH)])
    async with stdio_client(params) as (reader, writer):
        async with ClientSession(reader, writer) as session:
            await session.initialize()
            result = await session.call_tool(name, arguments)
            return result.content[0].text if result.content else ""


try:
    LOCATIONS_TEXT = await asyncio.wait_for(read_resource(LOCATIONS_URI), timeout=20)
except Exception as error:
    LOCATIONS_TEXT = f"reading the resource failed: {type(error).__name__}: {error}"

print(LOCATIONS_TEXT)

resources: ['file://locations.txt']
Africa/Abidjan
America/Halifax
America/New_York
America/Sao_Paulo
America/Toronto
America/Vancouver
Asia/Tokyo
Australia/Sydney
Europe/Berlin
Europe/Lisbon
Europe/London
Europe/Paris


In [49]:
check("w10-e1", LOCATIONS_TEXT)

✅ w10-e1 passed


True

## 2. A prompt: its name is not its title

**Context.** A prompt is a reusable instruction template, kept on the server so that every
client asking for the same task gets the same rules. It is chosen by the person, not the model.
This is exactly the shape session 10 packages as a skill: a template plus a note on when to use
it.

There is one trap, and the deck flags it. `@mcp.prompt(title="Timezone Conversion")` sets a
title for people to read. The prompt's **name** is the name of the function you decorated, and
the name is what the client asks for. The starter asks for the title.

**Instructions.**

1. Run both cells. The client prints the name the server lists next to the name you asked for.
2. Fix `PROMPT_NAME`.
3. Read the rendered text. The person's request is inside it, verbatim, which is the whole
   reason the template lives on the server rather than in the client.

**Expected output**

```
the server lists: ['convert_timezone_prompt']
you are asking for: 'convert_timezone_prompt'
name  = 'convert_timezone_prompt'
title = 'Timezone Conversion'

You are a timezone conversion engine.
...
User's timezone conversion request: It is 9:50 AM in the UK in January. What time is it in Lisbon, Portugal?
✅ w10-e2 passed
```

In [59]:
PROMPT = r'''

@mcp.prompt(title="Timezone Conversion")
def convert_timezone_prompt(timezone_request: str) -> str:
    """The timezone conversion task, its rules, and the person's request."""
    return f"""You are a timezone conversion engine.

Your task is to:
1. Extract the source datetime from the user's natural language input.
2. Identify the source timezone, explicit or inferred.
3. Convert the datetime into the target timezone.

Rules:
- If the date is ambiguous, resolve it against the provided datetime.
- If the input cannot be resolved confidently, seek clarification.

User's timezone conversion request: {timezone_request}"""
'''

write_server(SERVER_HEAD, TOOL, RESOURCE, PROMPT, MAIN)
print(f"rewrote {SERVER_PATH.name}: one tool, one resource, one prompt")

rewrote timezone_server.py: one tool, one resource, one prompt


In [60]:
USER_REQUEST = "It is 9:50 AM in the UK in January. What time is it in Lisbon, Portugal?"

PROMPT_NAME = "convert_timezone_prompt"  # <------ EDIT THIS LINE

try:
    PROMPT_ANSWER = await asyncio.wait_for(read_prompt(PROMPT_NAME, USER_REQUEST), timeout=20)
except Exception as error:
    PROMPT_ANSWER = (PROMPT_NAME, PROMPT_NAME, f"{type(error).__name__}: {error}")

print(f"name  = {PROMPT_ANSWER[0]!r}")
print(f"title = {PROMPT_ANSWER[1]!r}")
print()
print(PROMPT_ANSWER[2])

the server lists: ['convert_timezone_prompt']
you are asking for: 'convert_timezone_prompt'
name  = 'convert_timezone_prompt'
title = 'Timezone Conversion'

You are a timezone conversion engine.

Your task is to:
1. Extract the source datetime from the user's natural language input.
2. Identify the source timezone, explicit or inferred.
3. Convert the datetime into the target timezone.

Rules:
- If the date is ambiguous, resolve it against the provided datetime.
- If the input cannot be resolved confidently, seek clarification.

User's timezone conversion request: It is 9:50 AM in the UK in January. What time is it in Lisbon, Portugal?


In [61]:
check("w10-e2", PROMPT_ANSWER)

✅ w10-e2 passed


True

## 3. The router: a decision you can read

**Context.** The deck hands the tools to Claude and lets the model decide. That needs a key, and
a model's choice is not reproducible, so here the decision is a function. Two places named means
convert. A country with several zones means ask. A tool result in the message means write the
reply.

`FakeLLM` delivers those decisions through the same `complete(system, user)` call a real
provider takes, so the loop in the next cell is the loop you would write against Claude. Nothing
in this cell needs fixing. Read it, because exercise 4 turns on one branch of it.

In [62]:
import json
import re

from bootcamp_agent.llm import FakeLLM

REFERENCE_DATE = "2025-01-20"  # a January date: the recorded table knows no other month

#: Countries whose zones in the table number more than one. Written down rather
#: than derived, because a zone name does not say which country it is in.
COUNTRY_ZONES = {"canada": ("America/Halifax", "America/Toronto", "America/Vancouver")}

#: Places with exactly one zone in the table.
ALIASES = {
    "uk": "Europe/London",
    "britain": "Europe/London",
    "england": "Europe/London",
    "portugal": "Europe/Lisbon",
    "france": "Europe/Paris",
    "germany": "Europe/Berlin",
    "japan": "Asia/Tokyo",
    "brazil": "America/Sao_Paulo",
}

ZONE_NAMES = sorted(json.loads(FIXTURE.read_text(encoding="utf-8"))["zones"])


def _at(text, word):
    """Where `word` appears as a whole word, or -1. Whole word, so Paris is not Parish."""
    found = re.search(rf"\b{re.escape(word)}\b", text.lower())
    return found.start() if found else -1


def places_named(message):
    """The table's zones the message names, in the order it names them."""
    where = {}
    for zone in ZONE_NAMES:
        at = _at(message, zone.split("/")[-1].replace("_", " "))
        if at >= 0:
            where[zone] = at
    for alias, zone in ALIASES.items():
        at = _at(message, alias)
        if at >= 0:
            where[zone] = min(where.get(zone, at), at)
    return sorted(where, key=lambda zone: where[zone])


def country_named(message):
    """The name of a country with several zones in the table, or None."""
    return next((name for name in COUNTRY_ZONES if _at(message, name) >= 0), None)


def clock_named(message, default="09:50"):
    """The first clock time in the message, zero-padded."""
    found = re.search(r"\b(\d{1,2}):(\d{2})\b", message)
    return f"{int(found.group(1)):02d}:{found.group(2)}" if found else default


def _call(message, from_timezone, to_timezone):
    return json.dumps(
        {
            "tool": "convert_timezone",
            "arguments": {
                "date_time": f"{REFERENCE_DATE}T{clock_named(message)}:00",
                "from_timezone": from_timezone,
                "to_timezone": to_timezone,
            },
        }
    )


def decide(message, locations):
    """What a model would decide, as rules you can read. Returns one JSON object.

    A tool result in the message -> write the reply.
    A country with several zones -> offer the choices and ask.
    Two places named             -> call the tool.
    """
    if "Time in " in message:
        line = next(text for text in message.splitlines() if "Time in " in text)
        zone, converted = line.split("Time in ", 1)[1].split(": ", 1)
        return json.dumps({"answer": f"It is {converted} in {zone}."})
    country = country_named(message)
    if country:
        offered = [zone for zone in COUNTRY_ZONES[country] if zone in locations.splitlines()]
        if len(offered) >= 2:
            return json.dumps(
                {
                    "answer": f"{country.title()} has several time zones on this server: "
                    f"{', '.join(offered)}. Which one do you mean?"
                }
            )
        # Nothing to offer, because nothing told the router which zones this
        # server has. So it picks one and converts. That is exercise 4.
        return _call(message, "Europe/London", COUNTRY_ZONES[country][0])
    places = places_named(message)
    if len(places) >= 2:
        return _call(message, places[0], places[1])
    return json.dumps({"answer": "Which zone should I convert from, and which one to?"})


LLM = FakeLLM()  # the seam: one complete(system, user), offline and deterministic


def model(message, locations, system=""):
    """One model call. `decide` supplies the judgment, FakeLLM delivers it.

    A real model is one env var away: set BOOTCAMP_PROVIDER (SETUP.md) and replace
    these two lines with get_client(Settings.from_env()).complete(system, message).
    The loop below does not change when you do.
    """
    LLM.responses = {message: decide(message, locations)}
    return LLM.complete(system=system, user=message)


print(decide(USER_REQUEST, LOCATIONS_TEXT))
print(decide("Time in Europe/Lisbon: 2025-01-20T09:50:00+00:00", LOCATIONS_TEXT))

{"tool": "convert_timezone", "arguments": {"date_time": "2025-01-20T09:50:00", "from_timezone": "Europe/London", "to_timezone": "Europe/Lisbon"}}
{"answer": "It is 2025-01-20T09:50:00+00:00 in Europe/Lisbon."}


## 4. Five steps, and the one people leave out

**Context.** The deck's tool-calling workflow has five steps: the model sees the query and the
tools, it decides to call one, the tool runs, **the result goes back to the model**, and the
model writes the answer. Step four is the one that gets left out, and leaving it out is not
loud. The loop still runs. The model just answers a question it was never given the answer to.

The trace is the point of this exercise as much as the loop is. Session 9 is about reading one
when a loop misbehaves, and a five-step trace with a wrong answer is exactly the case where the
trace tells you more than the answer does.

**Instructions.**

1. Run both cells. Five steps come back and the final answer is a placeholder.
2. Look at the `tool_result` step. That text exists and the model never saw it.
3. Fix `followup` so the follow-up message carries the tool's result as well as the query.

**Expected output**

```
         llm  {"tool": "convert_timezone", "arguments": {"date_time": "2025-01-20T09:50:00", "from_tim
    tool_use  {"tool": "convert_timezone", "arguments": {"date_time": "2025-01-20T09:50:00", "from_tim
 tool_result  Time in Europe/Lisbon: 2025-01-20T09:50:00+00:00
         llm  {"answer": "It is 2025-01-20T09:50:00+00:00 in Europe/Lisbon."}
       final  It is 2025-01-20T09:50:00+00:00 in Europe/Lisbon.
✅ w10-e3 passed
```

9:50 in London is 9:50 in Lisbon. They share an offset in January, and the deck picks the pair
for exactly that reason: an answer that looks like no conversion happened is still the right
answer.

In [64]:
SYSTEM_PROMPT = PROMPT_ANSWER[2]  # the prompt's rules, fetched from the server above


async def run_loop(user_query, locations, system=""):
    """The deck's five steps, and a trace of what each one did.

    The deck concatenates the prompt and the locations into one user message. Here
    they are the system message and the request is the user turn: the same two
    pieces, in the two slots every provider gives you.
    """
    trace = []
    context = f"{system}\n\nSupported locations:\n{locations}".strip()

    raw = model(user_query, locations, system=context)  # 1. the model decides
    trace.append({"step": "llm", "detail": raw})
    decision = json.loads(raw)

    if "tool" in decision:  # 2. it chose a tool
        trace.append({"step": "tool_use", "detail": raw})
        tool_text = await asyncio.wait_for(
            call_tool(decision["tool"], decision["arguments"]), timeout=20
        )
        trace.append({"step": "tool_result", "detail": tool_text})  # 3. the tool ran

        followup = f"{user_query}\nTool result: {tool_text}" # <------ EDIT THIS LINE
        raw = model(followup, locations, system=context)  # 4. back to the model
        trace.append({"step": "llm", "detail": raw})
        decision = json.loads(raw)

    answer = decision.get(  # 5. the model writes the answer
        "answer", "(no answer: the model asked for the tool again, so it never saw the result)"
    )
    trace.append({"step": "final", "detail": answer})
    return trace


try:
    TRACE = await asyncio.wait_for(
        run_loop(USER_REQUEST, LOCATIONS_TEXT, system=SYSTEM_PROMPT), timeout=40
    )
except Exception as error:
    TRACE = [{"step": "failed", "detail": f"{type(error).__name__}: {error}"}]

for entry in TRACE:
    print(f"{entry['step']:>12}  {entry['detail'][:88]}")

         llm  {"tool": "convert_timezone", "arguments": {"date_time": "2025-01-20T09:50:00", "from_tim
    tool_use  {"tool": "convert_timezone", "arguments": {"date_time": "2025-01-20T09:50:00", "from_tim
 tool_result  Time in Europe/Lisbon: 2025-01-20T09:50:00+00:00
         llm  {"answer": "It is 2025-01-20T09:50:00+00:00 in Europe/Lisbon."}
       final  It is 2025-01-20T09:50:00+00:00 in Europe/Lisbon.


In [66]:
check("w10-e3", TRACE)

✅ w10-e3 passed


True

## 5. A question nobody can answer, answered anyway

**Context.** "What time is it in Canada?" names a country with three zones in the table. There
is no right conversion, and the deck's prompt says so in its own rules: *if the input cannot be
resolved confidently, seek clarification.* Clarifying needs one thing the model does not have on
its own, and that is the list of zones this server actually supports. The resource from exercise
1 is that list. This is what read-only context is **for**.

The starter passes no context. The router knows a country was named, has nothing to offer, and
so it converts. Read the answer it produces: it is a sentence with a time in it, and it is a
guess.

**Instructions.**

1. Run both cells. With exercise 3 fixed, the loop answers a request that has no answer
   with a converted time. Before that it ends on the placeholder from exercise 3.
2. Pass the locations resource in as the context.
3. Read the answer again. Refusal and clarification are outcomes, not failures, and
   `docs/guides/loop-engineering.md` designs them before the happy path.

**Expected output**

```
         llm  {"answer": "Canada has several time zones on this server: America/Halifax, America/Toron
       final  Canada has several time zones on this server: America/Halifax, America/Toronto, America/

Canada has several time zones on this server: America/Halifax, America/Toronto, America/Vancouver. Which one do you mean?
✅ w10-e4 passed
```

In [69]:
AMBIGUOUS_REQUEST = "What time is it in Canada?"

CANADA_CONTEXT = LOCATIONS_TEXT # <------ EDIT THIS LINE

try:
    CANADA_TRACE = await asyncio.wait_for(
        run_loop(AMBIGUOUS_REQUEST, CANADA_CONTEXT, system=SYSTEM_PROMPT), timeout=40
    )
except Exception as error:
    CANADA_TRACE = [{"step": "failed", "detail": f"{type(error).__name__}: {error}"}]

for entry in CANADA_TRACE:
    print(f"{entry['step']:>12}  {entry['detail'][:88]}")

CANADA_ANSWER = CANADA_TRACE[-1]["detail"]
print()
print(CANADA_ANSWER)

         llm  {"answer": "Canada has several time zones on this server: America/Halifax, America/Toron
       final  Canada has several time zones on this server: America/Halifax, America/Toronto, America/

Canada has several time zones on this server: America/Halifax, America/Toronto, America/Vancouver. Which one do you mean?


In [70]:
check("w10-e4", CANADA_ANSWER)

✅ w10-e4 passed


True

## What to take into unit 11

| You wrote | It became |
|---|---|
| `@mcp.resource(uri)` | read-only context, addressed by URI, fetched by the application |
| `@mcp.prompt(title=...)` | a template whose **name** is the function's, and whose title is for people |
| a five-step loop | a call the model can see the result of, and a trace you can read |
| a clarifying question | an outcome, rather than a converted time nobody asked for |

Unit 11 puts a database and an API behind the tool, and asks what you owe a server somebody else
wrote.

Run the scorecard.

In [71]:
review("w10")

w10: 4/4 passed  ·  400/400 marks


True